# ACS Income



In [ ]:
import pickle
import random

import pandas as pd
from scipy.stats import pearsonr

try:
    from experiment import (
        ROOT, environment_versions, experiment_config, prepare_acs,
        procedure_sha256, run_trials, save_results, summarize_results,
    )
    from plotting import plot_comparison, plot_qq_matrix, plot_target_qq
except ImportError:
    from dataset_ranking.experiment import (
        ROOT, environment_versions, experiment_config, prepare_acs,
        procedure_sha256, run_trials, save_results, summarize_results,
    )
    from dataset_ranking.plotting import (
        plot_comparison, plot_qq_matrix, plot_target_qq,
    )


## Prepare data

In [ ]:
DATA_DIR = ROOT / "data"
RESULT_DIR = ROOT / "results"
FIGURE_DIR = RESULT_DIR / "figures"

data = prepare_acs(DATA_DIR, target_state=6)
pd.DataFrame({"label": data["labels"], "state": data["source_ids"]})


## Run experiment

In [ ]:
n_t = 30
n_k = 1000
n_test = 1000
trials = 1000
global_seed = 123
jobs = 1

data.update({
    "trials": trials,
    "seed_start": global_seed,
    "n_test": n_test,
    "n_t": n_t,
    "n_k": n_k,
})

expected_config = experiment_config(
    data, global_seed, trials, n_test, n_t, n_k, None,
)
result_path = RESULT_DIR / "acs_income_results.pkl"
if result_path.exists():
    with open(result_path, "rb") as file:
        payload = pickle.load(file)
    assert payload["labels"] == data["labels"]
    assert payload["source_ids"] == data["source_ids"]
    assert payload["data_sha256"] == data["data_sha256"]
    assert payload["seeds"] == list(range(global_seed, global_seed + trials))
    assert payload["prediction_data"] == "standardized"
    assert payload["baseline_data"] == "standardized"
    assert payload["duc_data"] == "standardized_then_whitened"
    assert payload["code_sha256"] == expected_config["code_sha256"]
    assert payload["score_x_definition"] == expected_config["score_x"]
    assert all(
        payload["config"][key] == value
        for key, value in expected_config.items() if key != "environment"
    )
    assert all(
        result["run_config_sha256"] == procedure_sha256(payload["config"])
        for result in payload["results"]
    )
    results = payload["results"]
else:
    results = run_trials(
        data,
        trials=trials,
        seed_start=global_seed,
        jobs=jobs,
        n_test=n_test,
        n_t=n_t,
        n_k=n_k,
        checkpoint_path=RESULT_DIR / "acs_income_checkpoint.pkl",
        checkpoint_every=4,
    )
    save_results(data, results, RESULT_DIR)

score_x_values = [value for result in results for value in result["score_x_list"]]
assert len(results) == trials
assert all(0 <= value <= 1 for value in score_x_values)
summary = summarize_results(results, data["labels"])
summary

## QQ plots

In [ ]:
plot_target_qq(
    data["qq_data"],
    data["target_id"],
    data["qq_source_ids"],
    data["site_column"],
    data["response"],
    labels=data["qq_labels"],
    target_label="CA",
    title="Normal QQ Plots: California vs Other States",
    path=FIGURE_DIR / "qqplot_california_vs_other_states.png",
)

plot_qq_matrix(
    data["qq_data"],
    data["qq_source_ids"],
    data["site_column"],
    data["response"],
    labels=data["qq_labels"],
    title="Normal QQ Plots: ACS Income",
    path=FIGURE_DIR / "qqplot_matrix_acs_income.png",
)


## Result plot

In [ ]:
highlights = {
    "duc": {"NY", "NJ"},
    "neg_kl": {"NY"},
    "neg_score_x": {"NY", "NJ"},
}

plot_comparison(
    summary,
    "Comparison of methods for data selection: ACS Income",
    y_column="avg_rank",
    y_label="Average Rank by Test MSE (1 = Best)",
    highlight_map=highlights,
    path=FIGURE_DIR / "fig3.png",
)

trial_idx = random.Random(1).randrange(trials)
single_trial = summarize_results([results[trial_idx]], data["labels"])
no_highlights = {name: set() for name in highlights}
plot_comparison(
    single_trial,
    f"Comparison of data selection metrics: Trial {trial_idx}",
    y_column="weighted_mse",
    y_label="Test MSE",
    highlight_map=no_highlights,
    path=FIGURE_DIR / "ACS_one_trial.png",
)

summary_300 = summarize_results(results[:300], data["labels"])
plot_comparison(
    summary_300,
    "Comparison of methods for data selection: ACS Income (300 Trials)",
    y_column="avg_rank",
    y_label="Average Rank by Test MSE (1 = Best)",
    highlight_map=no_highlights,
    label_offsets={
        "neg_score_x": {
            "FL": (-8, -8),
            "GA": (8, 8),
            "NJ": (-8, 10),
            "NY": (-8, -10),
        },
    },
    path=FIGURE_DIR / "acs_300.png",
)


Code for Table S1


In [ ]:
table_s1_trials = 500
table_s1_counts = [300, 400, 500]

table_s1_data = dict(data)
table_s1_data.update({
    "trials": table_s1_trials,
    "seed_start": global_seed,
    "n_test": n_test,
    "n_t": n_t,
    "n_k": n_k,
    "n_target_mean": 50,
})
table_s1_config = experiment_config(
    table_s1_data, global_seed, table_s1_trials, n_test, n_t, n_k, 50,
)
table_s1_path = RESULT_DIR / "acs_target_mean_50_results.pkl"

if table_s1_path.exists():
    with open(table_s1_path, "rb") as file:
        table_s1_payload = pickle.load(file)
    assert table_s1_payload["labels"] == data["labels"]
    assert table_s1_payload["source_ids"] == data["source_ids"]
    assert table_s1_payload["data_sha256"] == data["data_sha256"]
    assert table_s1_payload["seeds"] == list(
        range(global_seed, global_seed + table_s1_trials)
    )
    assert table_s1_payload["prediction_data"] == "standardized"
    assert table_s1_payload["baseline_data"] == "standardized"
    assert table_s1_payload["duc_data"] == "standardized_then_whitened"
    assert table_s1_payload["code_sha256"] == table_s1_config["code_sha256"]
    assert table_s1_payload["score_x_definition"] == table_s1_config["score_x"]
    assert all(
        table_s1_payload["config"][key] == value
        for key, value in table_s1_config.items() if key != "environment"
    )
    assert all(
        result["run_config_sha256"] == procedure_sha256(table_s1_payload["config"])
        for result in table_s1_payload["results"]
    )
    table_s1_results = table_s1_payload["results"]
else:
    table_s1_results = run_trials(
        table_s1_data,
        trials=table_s1_trials,
        seed_start=global_seed,
        jobs=jobs,
        n_test=n_test,
        n_t=n_t,
        n_k=n_k,
        n_target_mean=50,
        checkpoint_path=RESULT_DIR / "acs_target_mean_50_checkpoint.pkl",
        checkpoint_every=4,
    )
    table_s1_output_dir = RESULT_DIR / "table_s1"
    save_results(table_s1_data, table_s1_results, table_s1_output_dir)
    (table_s1_output_dir / "acs_income_results.pkl").replace(table_s1_path)


In [ ]:
def table_s1_correlation(trials):
    trial_summary = summarize_results(
        table_s1_results[:trials], data["labels"]
    )
    return pearsonr(trial_summary["duc"], trial_summary["avg_rank"])[0]

table_s1 = pd.DataFrame({
    "trials": table_s1_counts,
    "correlation": [
        table_s1_correlation(trials) for trials in table_s1_counts
    ],
})
table_s1.to_csv(RESULT_DIR / "table_s1_correlations.csv", index=False)
table_s1_display = table_s1.assign(correlation=table_s1.correlation.round(3))
assert table_s1_display.correlation.tolist() == [-0.801, -0.812, -0.818]
table_s1_display


## Unequal source sizes


In [ ]:
try:
    from acs_unequal_source_sizes import (
        SOURCE_SIZE_BY_STATE, run_trials as run_unequal_trials,
        save_results as save_unequal_results, unequal_size_config,
    )
except ImportError:
    from dataset_ranking.acs_unequal_source_sizes import (
        SOURCE_SIZE_BY_STATE, run_trials as run_unequal_trials,
        save_results as save_unequal_results, unequal_size_config,
    )

unequal_output_dir = RESULT_DIR / "acs_unequal_source_sizes"
unequal_result_path = (
    unequal_output_dir / "acs_unequal_source_sizes_results.pkl"
)
source_sizes = [SOURCE_SIZE_BY_STATE[label] for label in data["labels"]]
unequal_config = unequal_size_config(
    data, source_sizes, trials, global_seed
)

if unequal_result_path.exists():
    with open(unequal_result_path, "rb") as file:
        unequal_payload = pickle.load(file)
    assert unequal_payload["labels"] == data["labels"]
    assert unequal_payload["source_ids"] == data["source_ids"]
    assert unequal_payload["source_sizes"] == source_sizes
    assert all(
        unequal_payload["config"][key] == value
        for key, value in unequal_config.items() if key != "environment"
    )
    assert all(
        result["run_config_sha256"]
        == procedure_sha256(unequal_payload["config"])
        for result in unequal_payload["results"]
    )
    unequal_results = unequal_payload["results"]
    unequal_summary = unequal_payload["summary"]
else:
    unequal_results, unequal_config = run_unequal_trials(
        data,
        source_sizes,
        trials=trials,
        seed_start=global_seed,
        jobs=jobs,
        checkpoint_path=unequal_output_dir / "checkpoint.pkl",
        checkpoint_every=4,
    )
    unequal_summary, unequal_correlations = save_unequal_results(
        data, source_sizes, unequal_results, unequal_config,
        unequal_output_dir,
    )

unequal_summary


In [ ]:
plot_comparison(
    unequal_summary,
    "Comparison of methods for data selection: ACS Income\n"
    "Unequal Source Sizes",
    y_column="avg_rank",
    y_label="Average Rank of Test MSE (1 = Best)",
    highlight_map={
        "duc": {"IL"},
        "neg_kl": {"IL"},
        "neg_score_x": {"IL"},
    },
    red_highlight_map={
        "duc": {"GA"},
        "neg_kl": {"GA"},
        "neg_score_x": {"GA"},
    },
    corr_y=0.86,
    label_offsets={
        "duc": {
            "GA": (8, 0), "NC": (8, 0),
            "NJ": (-8, -12), "NY": (-8, 12),
        },
        "neg_kl": {"GA": (8, 0), "NC": (8, -8)},
        "neg_score_x": {
            "GA": (8, 0), "NC": (8, -8),
            "NJ": (-8, 0), "NY": (-8, 12),
        },
    },
    path=FIGURE_DIR / "acs_unequal_source_sizes.png",
)
